# Übersetzungs-Test RM ⇄ DE (Colab)

Modell: [ZurichNLP/romansh-nllb-1.3b-ct2](https://huggingface.co/ZurichNLP/romansh-nllb-1.3b-ct2) · Lizenz CC-BY-NC-4.0 (nicht kommerziell)

**Nur mit unkritischem Testtext verwenden.** Der Text wird auf einem Google-Server verarbeitet, nicht auf dem eigenen Rechner.

**Ablauf**
1. Menü *Laufzeit → Laufzeittyp ändern → T4-GPU* wählen (viel schneller; ohne GPU geht es auch, dauert aber länger).
2. *Laufzeit → Alle ausführen*. Beim ersten Mal dauert Zelle 1–2 ein paar Minuten (Installation, Modell-Download ca. 1.3 GB).
3. In Zelle 3 den Text zwischen die drei Anführungszeichen einfügen (eine Frase pro Zeile, z. B. aus dem Studio-Test), Richtung und Idiom wählen und die Zelle mit ▶ ausführen.
4. Ergebnis kopieren und im Studio-Test in Schritt 3 einfügen.

In [ ]:
#@title 1 · Installieren
!pip -q install ctranslate2 sentencepiece huggingface_hub transformers nvidia-cublas-cu12


In [ ]:
#@title 2 · Modell laden
import ctranslate2, time
from huggingface_hub import snapshot_download
from transformers import NllbTokenizer
t0 = time.time()
path = snapshot_download("ZurichNLP/romansh-nllb-1.3b-ct2")
tok = NllbTokenizer.from_pretrained(path)
dev = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
tr = ctranslate2.Translator(path, device=dev)
print(f"Modell bereit auf {dev.upper()} ({time.time()-t0:.0f} s)")

def translate(src, tgt, lines, bs=16):
    tok.src_lang = src
    out = []
    for i in range(0, len(lines), bs):
        part = lines[i:i+bs]
        batch = [tok.convert_ids_to_tokens(tok.encode(t)) for t in part]
        res = tr.translate_batch(batch, target_prefix=[[tgt]]*len(batch), beam_size=4, max_decoding_length=256)
        out += [tok.decode(tok.convert_tokens_to_ids(r.hypotheses[0][1:]), skip_special_tokens=True) for r in res]
    return out

import ctypes, glob, site
def preload_cublas():
    for sp in site.getsitepackages():
        for pat in ("nvidia/cublas/lib/libcublasLt.so.12*", "nvidia/cublas/lib/libcublas.so.12*"):
            for f in glob.glob(f"{sp}/{pat}"):
                try: ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
                except OSError: pass
preload_cublas()
try:
    translate("deu_Latn", "rm-rumgr", ["Test."]); print("GPU funktioniert ✓")
except Exception as e:
    print("GPU nicht nutzbar, wechsle auf CPU (langsamer):", str(e)[:120])
    tr = ctranslate2.Translator(path, device="cpu"); print("CPU bereit")


In [ ]:
# 3 · Übersetzen – Richtung und Idiom wählen, Text unten zwischen die """ einfügen (eine Frase pro Zeile), dann ▶ drücken
RICHTUNG = "RM → DE" #@param ["RM → DE", "DE → RM"]
IDIOM = "Rumantsch Grischun" #@param ["Rumantsch Grischun", "Sursilvan", "Sutsilvan", "Surmiran", "Puter", "Vallader"]
TEXT = """
Buna saira, chars spectaturs.
Oz discurrin nus davart la neiv.
"""

import html
from IPython.display import HTML, display
IDIOMS = {"Rumantsch Grischun":"rm-rumgr","Sursilvan":"rm-sursilv","Sutsilvan":"rm-sutsilv","Surmiran":"rm-surmiran","Puter":"rm-puter","Vallader":"rm-vallader"}
rm = IDIOMS[IDIOM]
src, tgt = (rm, "deu_Latn") if RICHTUNG == "RM → DE" else ("deu_Latn", rm)
lines = [l.strip() for l in TEXT.strip().splitlines()]
idx = [i for i, l in enumerate(lines) if l]          # leere Zeilen bleiben leer (Zeilenzahl stimmt)
res = translate(src, tgt, [lines[i] for i in idx])
full = [""] * len(lines)
for i, r in zip(idx, res): full[i] = r.replace("\n", " ").strip()
result = "\n".join(full)
print(f"✓ {len(idx)} Zeilen übersetzt ({RICHTUNG}, {IDIOM}) – bitte prüfen:\n")
print(result)
display(HTML('<textarea id="res" style="width:100%;height:200px;font:14px sans-serif">' + html.escape(result) + '</textarea><br>'
  '<button onclick="const t=document.getElementById(\'res\');t.select();document.execCommand(\'copy\');this.textContent=\'Kopiert ✓\'">Ergebnis kopieren</button>'))
